# Basic implementation for the CKKS algorithm using Pyphel

## Author: José Ángel de Bustos Pérez
## License: GNU General Public License v3.0

This notebook uses [Pyphel](https://pyfhel.readthedocs.io/) to implement the CKKS homomorphic encryption algorithm.

CKKS (Cheon–Kim–Kim–Song, 2017) is a homomorphic encryption scheme based on RLWE (Ring Learning With Errors), like BFV, but designed to work with real or complex numbers in an approximate way. That means that the operations will add some error.

Key differences compared to BFV:

* Data: float/double, not integers.
* Results: approximate (there is rounding error).
* Rescaling: after each multiplication, rescaling is required.
* Levels: each rescale consumes a level of the ciphertext.
* Slots: n/2 instead of n (slots are complex conjugates).

It is the standard scheme for machine learning on encrypted data, statistics, signal processing, etc.

The first step is to set the parameters and generate the keys:

* **n**, polynomial degree (power of 2). In CKKS, the number of slots is n/2.
* **scale**, scaling factor. CKKS encodes floats by multiplying them by this factor to convert them into large integers. A larger scale gives more decimal precision but adds more noise. Typical scale value used: 2^30.
* **qi_sizes**, sizes (in bits) of the prime moduli that form the modulus chain. CKKS uses modulus switching: each rescale after a multiplication removes one prime from the chain. The number of intermediate primes equals the multiplicative depth you can support.
* **sec**, security level in bits.

In [1]:
from Pyfhel import Pyfhel
import numpy as np

HE = Pyfhel()

ckks_params = {
    'scheme': 'CKKS',
    'n': 2**14,                # 16384 coefficients => 8192 slots for batching
    'scale': 2**30,            # approximate precision: ~9 decimal places
    # qi_sizes gives a chain of 7 primes. The first and last are “large” (60 bits) for technical reasons; 
    # the ones in the middle are the size of the scale (30 bits). ⇒ Approximately 5 levels of multiplication.
    'qi_sizes': [60, 30, 30, 30, 30, 30, 60],
    'sec': 128,
}

HE.contextGen(**ckks_params)
HE.keyGen()           # Key creation
HE.rotateKeyGen()     # Required for rotations (we will use it in batching)
HE.relinKeyGen()      # Required to reduce size after multiplication

print(f"Scheme: {HE.scheme}")
print(f"n (Polynomial degree): {HE.n}")
print(f"Available Slots (n/2): {HE.n // 2}")
print(f"Scale: 2^{int(np.log2(HE.scale))}")
print(f"qi modulus chain: {ckks_params['qi_sizes']}")
print(f"=> Available multiplicative depth: " f"{len(ckks_params['qi_sizes']) - 2}")

ModuleNotFoundError: No module named 'Pyfhel'

Unlike BFV, in CKKS the **noise budget** is not measured in the same way. What matters is the LEVEL of the ciphertext (how many primes remain in the modulus chain). Each rescale consumes one level. When you reach the last level, you can no longer perform multiplications.

## Basic operations

We will perform basic operations such as addition and multiplications to check how noise is increasing. We will start with addition:

In [2]:
import random

# Randon number generation
a = random.random()
b = random.random()

expected_value = a + b

# Encrypting data. CKKS always works with vectors. To encrypt a scalar, we place it
# in an array (the remaining slots are filled with zeros).
fhe_a = HE.encryptFrac(np.array([a], dtype=np.float64))
fhe_b = HE.encryptFrac(np.array([b], dtype=np.float64))

# Homomorphic operation (addition)
fhe_suma = fhe_a + fhe_b

# Getting the operation value, decrypting
suma = HE.decryptFrac(fhe_suma)[0]

# Error
error = abs(expected_value - suma)

print(f"Plaintext data: a = {a}, b = {b}")
print(f"Expected value: {expected_value}")
print(f"Homomorphic addition: {suma}")
print(f"Error: {error:.2e}")

Plaintext data: a = 0.2546509554181008, b = 0.6184822838021392
Expected value: 0.87313323922024
Homomorphic addition: 0.87313525611518
Error: 2.02e-06


We are going to proceed with multiplication.

In [3]:
expected_value = a * b

# Homomorphic multiplication
fhe_mult = fhe_a * fhe_b
# Relinearisation
~fhe_mult

# Rescalation. After each multiplication, we must manually rescale. CKKS multiplies two numbers scaled by 
# S, and the result ends up scaled by S^2. The rescale operation divides by S and removes one prime from the 
# chain to keep the original scaling factor S
HE.rescale_to_next(fhe_mult)

# Getting the operation value, decrypting
mult = HE.decryptFrac(fhe_mult)[0]

# Error
error = abs(expected_value - mult)

print(f"Plaintext data: a = {a}, b = {b}")
print(f"Expected value: {expected_value}")
print(f"Homomorphic multiplication: {mult}")
print(f"Error: {error:.2e}")

Plaintext data: a = 0.2546509554181008, b = 0.6184822838021392
Expected value: 0.1574971044793837
Homomorphic multiplication: 0.15749678533188546
Error: 3.19e-07


## Performing homomorphic encryption with multiple operations

We are going to show how to perform more complex operations with homomorphic encryption. Let's assume we want to perform homomorphic encryption to:

$$f(x,y) = (x+y)^2 - (x-y)^2$$

In [4]:
# Random number generation
x = random.random()
y = random.random()

expected_value = (x + y)**2 - (x-y)**2

# Encrypting data
fhe_x = HE.encryptFrac(np.array([x], dtype=np.float64))
fhe_y = HE.encryptFrac(np.array([y], dtype=np.float64))

# (x + y)
fhe_addition_xy = fhe_x + fhe_y

# (x + y)^2
fhe_square_add = fhe_addition_xy * fhe_addition_xy
# Relinearisation
~fhe_square_add
# Rescaling, uses one prime in the modulus chain
HE.rescale_to_next(fhe_square_add)

# (x - y)
fhe_substraction_xy = fhe_x - fhe_y

# (x - y)^2
fhe_square_substraction = fhe_substraction_xy * fhe_substraction_xy
# Relinearisation
~fhe_square_substraction
# Rescaling, uses one prime in the modulus chain
HE.rescale_to_next(fhe_square_substraction)

# Final. Before doing the substraction both ciphertext must have the same
# level, in both we have performed one rescale operation so they are at the
# same level.
fhe_final = fhe_square_add - fhe_square_substraction

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_final)[0]

# Error
error = abs(expected_value - final)

print(f"Operation: ({x} + {y})**2 - ({x}-{y})**2")
print(f"Expected value: {expected_value}")
print(f"Homomorphic value: {final}")
print(f"Error: {error:.2e}")

Operation: (0.7770894469235451 + 0.6738939048225632)**2 - (0.7770894469235451-0.6738939048225632)**2
Expected value: 2.0947033671348545
Homomorphic value: 2.094711223414748
Error: 7.86e-06


Added noise is mostly the same that the noise added by the homomorphic multiplication. Now, we will perform the following:

$$f(x,y) = (x+y)^2 - (x-y)^3$$

In this example $(x+y)^2$ has a multiplicativity depth of 1 but $(x-y)^3$ has a multiplicativity depth of 2. That means that we will need to align levels (working on the same scale) to operate with them.

In [5]:
expected_value_third = (x + y)**2 - (x - y)**3

# Aligning data on the same scale. fhe_substraction_xy is in level 1
# but fhe_square_substraction is in level 2
fhe_substraction_xy_aligned = fhe_substraction_xy.copy()
HE.mod_switch_to_next(fhe_substraction_xy_aligned)

# (x - y)^3
fhe_third_substraction = fhe_square_substraction * fhe_substraction_xy_aligned
# Relinearisation
~fhe_third_substraction
# Rescaling, uses one prime in the modulus chain
HE.rescale_to_next(fhe_third_substraction)

# (x + y)^2 is in level 1 (one rescaling) but (x - y)^3 is in level 2
# before operating them, we need to have both of them on the  same level
fhe_square_add_aligned = fhe_square_add.copy()
HE.mod_switch_to_next(fhe_square_add_aligned)

# Final
fhe_final = fhe_square_add_aligned - fhe_third_substraction

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_final)[0]

# Error
error = abs(expected_value_third - final)

print(f"Operation: ({x} + {y})**2 - ({x}-{y})**3")
print(f"Expected value: {expected_value_third}")
print(f"Homomorphic value: {final}")
print(f"Error: {error:.2e}")

Operation: (0.7770894469235451 + 0.6738939048225632)**2 - (0.7770894469235451-0.6738939048225632)**3
Expected value: 2.104253724703301
Homomorphic value: 2.1044546867791674
Error: 2.01e-04


## Batching operation

Batching allows us to perform the same homomorphic operation to several data at the same time, parallelism.

If we are familiar with processor architectures we will know what **SIMD** (**S**imple **I**nstruction **M**ultiple **D**ata) is. One single operation using only one clock cicle operate on multiple data. Batching is the same but used in homomorphic encryption and it is used to speed up operations.

In [6]:
# Randon number generation
size = 10 # number of floats to operate at the same time
v1 = np.random.uniform(-100,  100, (size))
v2 = np.random.uniform(-100,  100, (size))

# Encrypting data
fhe_v1 = HE.encryptFrac(v1)
fhe_v2 = HE.encryptFrac(v2)

# Item by item homomorphic addition
fhe_add_vectorial = fhe_v1 + fhe_v2

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_add_vectorial)[:size]

# Error
expected_value = v1 + v2
error = np.abs(final - expected_value)

print(f"v1: {v1}")
print(f"v2: {v2}")
print(f"Homomorphic v1 + v2:  {final}")
print(f"Expected value: {expected_value}")
print(f"Error: {error}")

v1: [-86.19112623  92.63778315 -40.74631418  -1.27232174 -76.02763053
 -66.64733601 -75.20544121  38.6023865    4.62978384  43.82028829]
v2: [ 91.07707475  63.99018525  13.35348267  41.8841672   81.7170209
 -64.04491919 -93.17882516  40.73493852   9.0766509   56.72678748]
Homomorphic v1 + v2:  [   4.88595124  156.62797518  -27.3928311    40.61184562    5.68939009
 -130.69225304 -168.3842678    79.33732804   13.70643937  100.54707583]
Expected value: [   4.88594852  156.6279684   -27.39283151   40.61184546    5.68939037
 -130.6922552  -168.38426637   79.33732502   13.70643474  100.54707578]
Error: [2.71859921e-06 6.77485698e-06 4.18722031e-07 1.60115121e-07
 2.78972951e-07 2.15911143e-06 1.42912793e-06 3.02039078e-06
 4.62944928e-06 5.23577768e-08]


Let's see what happens with batching multiplication:

In [7]:
# Item by item homomorphic multiplication
fhe_multiplication_vectorial = fhe_v1 * fhe_v2
# Relinearisation
~fhe_multiplication_vectorial

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_multiplication_vectorial)[:size]

# Error
expected_value = v1 * v2
error = np.abs(final - expected_value)

print(f"\nHomomorphic v1 * v2  = {final}")
print(f"Expected value = {v1 * v2}")
print(f"Error: {error}")


Homomorphic v1 * v2  = [-7850.03584879  5927.90944696  -544.10500598   -53.29014031
 -6212.75161159  4268.42310652  7007.5547714   1572.46595853
    42.02295499  2485.78418581]
Expected value = [-7850.03564667  5927.90890491  -544.10520009   -53.29013642
 -6212.75147359  4268.42324871  7007.55465758  1572.46584073
    42.02293167  2485.78418144]
Error: [2.02115393e-04 5.42051864e-04 1.94113882e-04 3.89480871e-06
 1.38001663e-04 1.42190806e-04 1.13817582e-04 1.17800610e-04
 2.33213852e-05 4.37158769e-06]


## Dot product or scalar product

Dot product or scalar product is a fundamental operation used in multiple algorithms such as linear regresion, support vector machines (SVM) or neural network algorithms. For this reason been able to operate it using homomorphic encription will ease using such algoritms with homomorphic encryption.

In [8]:
# Random sample
size = 10 # random sample size
v1 = np.random.uniform(-100,  100, (size))
v2 = np.random.uniform(-100,  100, (size))

# Encrypt data
fhe_v1 = HE.encryptFrac(v1)
fhe_v2 = HE.encryptFrac(v2)

# Cdot operation with encrypted data
fhe_cdot = fhe_v1 * fhe_v2
# Relinearisation
~fhe_cdot

# vector with product, component to component
value_cdot = HE.decryptFrac(fhe_cdot)[:size]

# Error
expected_value = float(np.dot(v1, v2))
error = abs(expected_value - np.sum(value_cdot))

print(f"v1: {v1}")
print(f"v2: {v2}")
# cdot product is the sum for all the values in value_cdot
print(f"Homomorphic v1 * v2: {np.sum(value_cdot)}")
print(f"Expected value: {expected_value}")
print(f"Error: {error:.2e}")

v1: [-85.54157414   6.66273899   6.31785316 -43.04074122 -58.58349498
 -65.24334803  -1.30760021  25.86996475  63.40055635  66.44103566]
v2: [ 58.69761219  -9.01607985 -63.40350172 -17.77949555  50.94774502
  -4.17214728  34.19476055 -98.63292643  48.34758844  42.33897224]
Homomorphic v1 * v2: -4147.01564083114
Expected value: -4147.015623129597
Error: 1.77e-05


## How many operations can be done before data is corrupted?

We have seen that the multiplication operation adds noise to the data. For this reason is important to know how many operations can be done without corrupting data. This number  of operations is the operational limit which indicates the maximum number of operations that can be performed by the algorithm.

In CKKS the limit is the prime number chain defined in **qi_sizes = [60, 30, 30, 30, 30, 30, 60]**. With each rescaling operation one prime number is used, in this case we have five multiplication levels

In [9]:
from random import randrange

data = randrange(10)
fhe_data = HE.encryptFrac(np.array([data], dtype=np.float64))

print(f"Encrypting data = {data}")
print(f"Initial qi_sizes: {ckks_params['qi_sizes']}")
print(f"=> Available multiplicative depth: " f"{len(ckks_params['qi_sizes']) - 2}")

# Start multiplying encrypted data
for i in range(1, 10):
    try:
        fhe_data = fhe_data * fhe_data   
        # Relinearisation
        ~fhe_data
        # Rescaling
        HE.rescale_to_next(fhe_data)
        expected_value = data ** (2 ** i)
        fhe_value = HE.decryptFrac(fhe_data)[0]
        error = abs(expected_value - fhe_value)
        print(f"Iteration {i}: data^{2**i:<5} | "
              f"Expected value={expected_value:<20.6f} "
              f"Encrypted value={fhe_value:<20.6f} | "
              f"Error={error:.2e}")
    except Exception as e:
        print(f"\nIteration {i}: FALLO - no more multiplication levels available.")
        print(f"  Excepción: {type(e).__name__}: {e}")
        break

Encrypting data = 8
Initial qi_sizes: [60, 30, 30, 30, 30, 30, 60]
=> Available multiplicative depth: 5
Iteration 1: data^2     | Expected value=64.000000            Encrypted value=64.000017            | Error=1.66e-05
Iteration 2: data^4     | Expected value=4096.000000          Encrypted value=4096.002123          | Error=2.12e-03
Iteration 3: data^8     | Expected value=16777216.000000      Encrypted value=16777233.390440      | Error=1.74e+01
Iteration 4: data^16    | Expected value=281474976710656.000000 Encrypted value=281475560234606.750000 | Error=5.84e+08
Iteration 5: data^32    | Expected value=79228162514264337593543950336.000000 Encrypted value=-53693131656.177917  | Error=7.92e+28

Iteration 6: FALLO - no more multiplication levels available.
  Excepción: ValueError: scale out of bounds
